 # ULTRON Training Control

Portable browser launcher for Kaggle, Google Colab, Lightning AI and GitHub Actions. It downloads ULTRON's own training engine, data generator, verifier, Transformer and checkpoint code.

**Safe validation mode is the default.** Run the validation cells to check the runtime, model architecture and generated-data verifier. A separate optional read-only connection test verifies the private Supabase training API and checkpoint Storage. Neither validation nor connection testing updates model weights.

On Kaggle, select GPU and enable Internet. The Kaggle API token (`kaggle.json`) is not needed inside this notebook: it is for Kaggle's external API/CLI. The notebook reads project files from the public GitHub repository. Private backend credentials belong only in Kaggle Secrets.

In [ ]:
import json
import pathlib
import subprocess
import sys
import urllib.parse
import urllib.request

REPOSITORY = "shayaanfarsheed-cmd/ULTRONAI"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}"
PROJECT_DIR = pathlib.Path.cwd() / ".ultron_training_control"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
FILES = ("train.py", "ultron_model.py", "ultron_data.py", "ultron_verify.py", "ultron_checkpoint.py", "ultron_training.py", "ultron_supabase.py", "ultron_storage.py", "ultron_session.py", "config.json", "servers.json", "requirements.txt")

for filename in FILES:
    request = urllib.request.Request(
        f"{BASE_URL}/{filename}",
        headers={"User-Agent": "ULTRON-Training-Control"},
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        final_host = urllib.parse.urlparse(response.geturl()).hostname
        if final_host != "raw.githubusercontent.com":
            raise RuntimeError("Unexpected download host; refusing to save the project file.")
        content = response.read(2_000_001)
    if len(content) > 2_000_000:
        raise RuntimeError(f"Project file is unexpectedly large: {filename}")
    if filename.endswith(".json"):
        json.loads(content.decode("utf-8"))
    (PROJECT_DIR / filename).write_bytes(content)

try:
    import numpy
    import safetensors
    import huggingface_hub
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

print("ULTRON project files retrieved from GitHub.")
print("Project folder:", PROJECT_DIR)

In [ ]:
command = [
    sys.executable,
    str(PROJECT_DIR / "train.py"),
    "--preflight",
    "--config",
    str(PROJECT_DIR / "config.json"),
    "--servers",
    str(PROJECT_DIR / "servers.json"),
]
result = subprocess.run(command, cwd=PROJECT_DIR, check=False)
if result.returncode != 0:
    raise RuntimeError(f"ULTRON preflight failed with exit code {result.returncode}")

summary_command = command + ["--model-summary"]
summary_result = subprocess.run(summary_command, cwd=PROJECT_DIR, check=False)
if summary_result.returncode != 0:
    raise RuntimeError(f"ULTRON model summary failed with exit code {summary_result.returncode}")

data_command = command + ["--data-smoke-test"]
data_result = subprocess.run(data_command, cwd=PROJECT_DIR, check=False)
if data_result.returncode != 0:
    raise RuntimeError(f"ULTRON data verification failed with exit code {data_result.returncode}")

print("Validation complete. No optimizer steps, checkpoint writes, or training sessions were run.")

## Private backend connection (read-only)

Add `SUPABASE_SECRET_KEY` and `SUPABASE_OWNER_ID` to Kaggle Secrets. Do not paste either value into a notebook cell, GitHub, or chat. Set `ENABLE_BACKEND_CONNECTION_TEST = True` in the next cell and run it. This performs owner-scoped reads against the private `training_api` schema and lists the owner's checkpoint prefix. It does not create a run, acquire a lease, write a checkpoint, or update model weights. Leave `ENABLE_REAL_TRAINING = False`.

In [ ]:
import os
import uuid

ENABLE_BACKEND_CONNECTION_TEST = False  # Set True to test the private dashboard backend; this does not train.
BACKEND_CONNECTION_VERIFIED = False
ENABLE_REAL_TRAINING = False  # Keep False until the training pipeline and frontend readiness gates pass.

def _read_platform_secret(name):
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if ENABLE_BACKEND_CONNECTION_TEST or ENABLE_REAL_TRAINING:
    config = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))
    os.environ["SUPABASE_URL"] = config["database"]["project_url"]
    for secret_name in ("SUPABASE_SECRET_KEY", "SUPABASE_OWNER_ID"):
        secret_value = _read_platform_secret(secret_name)
        if not secret_value:
            raise RuntimeError("Add SUPABASE_SECRET_KEY and SUPABASE_OWNER_ID to Kaggle Secrets. Never paste them into notebook code.")
        os.environ[secret_name] = secret_value
    if not os.environ["SUPABASE_SECRET_KEY"].startswith("sb_secret_"):
        raise RuntimeError("Use a current Supabase secret key in Kaggle Secrets; never use a publishable key.")
    uuid.UUID(os.environ["SUPABASE_OWNER_ID"])

    if ENABLE_BACKEND_CONNECTION_TEST:
        connection_command = [
            sys.executable,
            str(PROJECT_DIR / "train.py"),
            "--connection-test",
            "--config", str(PROJECT_DIR / "config.json"),
            "--servers", str(PROJECT_DIR / "servers.json"),
        ]
        connection_result = subprocess.run(connection_command, cwd=PROJECT_DIR, check=False)
        if connection_result.returncode != 0:
            raise RuntimeError("ULTRON backend connection test failed. No training was started.")
        BACKEND_CONNECTION_VERIFIED = True
        print("ULTRON private backend connection verified. No records or checkpoints were written.")

    if ENABLE_REAL_TRAINING:
        hf_token = _read_platform_secret("HF_TOKEN")
        hf_repo_id = _read_platform_secret("HF_REPO_ID")
        if bool(hf_token) != bool(hf_repo_id):
            raise RuntimeError("Add both HF_TOKEN and HF_REPO_ID, or leave both unset.")
        if hf_token and hf_repo_id:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["HF_REPO_ID"] = hf_repo_id
        print("Private credentials loaded. Values are not displayed.")
else:
    print("Backend connection test and training are disabled. No private credentials were requested or loaded.")


## Real training (disabled)

The notebook training cell remains disabled. A successful backend connection test proves only that Kaggle can reach the private status API and checkpoint storage; it does not validate the complete training loop or authorize a real weight-update session. Keep `ENABLE_REAL_TRAINING = False` until the full training/checkpoint/evaluation path and frontend readiness are validated and the owner explicitly approves starting training.

In [ ]:
if not ENABLE_REAL_TRAINING:
    print("ULTRON training is disabled. No model weights were updated.")
elif not BACKEND_CONNECTION_VERIFIED:
    raise RuntimeError("Run the private backend connection test successfully first. Training was not started.")
else:
    training_command = [
        sys.executable,
        str(PROJECT_DIR / "train.py"),
        "--train",
        "--config",
        str(PROJECT_DIR / "config.json"),
        "--servers",
        str(PROJECT_DIR / "servers.json"),
    ]
    training_result = subprocess.run(training_command, cwd=PROJECT_DIR, check=False)
    if training_result.returncode != 0:
        raise RuntimeError(f"ULTRON training session stopped with exit code {training_result.returncode}")
